# 03 — Patches, Tokens, Spectrograms, and Budgeting

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/03_patches_tokens_spectrograms_and_budgeting.ipynb)

**Prerequisites**: [01](01_intro_to_multimodal_systems.ipynb), [02](02_model_families_and_modality_interfaces.ipynb)

**What you will learn**:
- Image patching from first principles: how pixels become tokens
- Dynamic resolution: why modern VLMs handle arbitrary image sizes
- Audio spectrograms: waveform → mel spectrogram → embeddings
- Video frame sampling strategies: uniform, keyframe, scene-change
- A unified token budget calculator for any multimodal workload

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Image Patches from First Principles**
- Understand **Dynamic Resolution**
- Understand **Audio Spectrograms**
- Understand **Video Frame Sampling**
- Understand **The Unified Token Budget Calculator**


In [ ]:
# --- Setup ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow

import json, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter, defaultdict

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Dependencies ready.")

## 1 — Image Patches from First Principles

Vision Transformers (ViT) introduced the idea: **split an image into a grid of patches, treat each patch like a token**.

For a 224×224 image with 14×14 patches:
- Grid: 16×16 = 256 patches
- Each patch is flattened into a vector (14×14×3 = 588 values)
- A linear projection maps this to the model’s hidden dimension (e.g., 768)

The result: 256 "visual tokens" that enter the transformer alongside text tokens.

In [ ]:
# Implement image patching from scratch
def extract_patches(image_array, patch_size=14):
    H, W, C = image_array.shape
    assert H % patch_size == 0 and W % patch_size == 0
    n_h, n_w = H // patch_size, W // patch_size
    patches = []
    for i in range(n_h):
        for j in range(n_w):
            patch = image_array[i*patch_size:(i+1)*patch_size,
                               j*patch_size:(j+1)*patch_size, :]
            patches.append(patch.flatten())
    return np.array(patches), (n_h, n_w)

# Create a test image with recognizable features
img = np.zeros((224, 224, 3), dtype=np.uint8)
img[50:174, 50:174] = [200, 50, 50]     # Red square
img[80:144, 80:144] = [50, 50, 200]     # Blue inner square
img[20:40, 20:204] = [50, 200, 50]      # Green bar at top

patches, grid = extract_patches(img, patch_size=14)
print(f"Image shape: {img.shape}")
print(f"Patch grid: {grid[0]}x{grid[1]} = {len(patches)} patches")
print(f"Each patch vector: {patches.shape[1]} values (14x14x3)")

# Visualize patches
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img)
axes[0].set_title("Original (224x224)")
axes[0].axis("off")

# Show patch grid
axes[1].imshow(img)
for i in range(0, 225, 14):
    axes[1].axhline(y=i, color="white", linewidth=0.3)
    axes[1].axvline(x=i, color="white", linewidth=0.3)
axes[1].set_title(f"{len(patches)} patches (14x14)")
axes[1].axis("off")

# Show individual patches as a montage
montage = patches[:64].reshape(8, 8, 14, 14, 3).transpose(0, 2, 1, 3, 4).reshape(112, 112, 3)
axes[2].imshow(montage)
axes[2].set_title("First 64 patches (8x8 montage)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 2 — Dynamic Resolution

Early ViTs required fixed 224×224 input. Modern VLMs like Qwen2-VL handle **arbitrary resolutions** by:

1. Dividing the image into tiles (e.g., 448×448 each)
2. Patching each tile independently
3. Adding special tokens to mark tile boundaries

This means **token count scales with resolution**:

| Resolution | Tiles (448×448) | Patches/tile | Total patches |
|------------|-------------------|-------------|---------------|
| 448×448 | 1 | 1,024 | 1,024 |
| 896×896 | 4 | 1,024 | 4,096 |
| 1344×1344 | 9 | 1,024 | 9,216 |

**Practical impact**: A high-res document image can consume 4–9× more tokens than a thumbnail.

In [ ]:
# Token count vs resolution
resolutions = [(224, 224), (448, 448), (672, 672), (896, 896), (1120, 1120), (1344, 1344)]
patch_size = 14

res_labels = []
token_counts = []
for h, w in resolutions:
    n = (h // patch_size) * (w // patch_size)
    res_labels.append(f"{h}x{w}")
    token_counts.append(n)
    print(f"  {h:5d}x{w:<5d} -> {n:6,} patches")

plt.figure(figsize=(8, 4))
plt.bar(res_labels, token_counts, color="#4CAF50")
plt.ylabel("Number of visual tokens")
plt.title("Visual Token Count vs Image Resolution (patch_size=14)")
for i, v in enumerate(token_counts):
    plt.text(i, v + 100, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()
print("Doubling resolution -> 4x tokens (quadratic scaling)")

## 3 — Audio Spectrograms

Audio processing converts time-domain waveforms into frequency-domain representations:

1. **Waveform**: Raw audio samples (e.g., 16,000/sec for speech)
2. **STFT**: Short-Time Fourier Transform splits into overlapping windows
3. **Mel scale**: Map frequencies to perceptual scale (humans hear logarithmically)
4. **Log compression**: Apply log to match human loudness perception

The result is a **mel spectrogram**: a 2D image where x=time, y=frequency, brightness=energy.

In [ ]:
# Build a mel spectrogram from first principles (simplified)
sr = 16000
duration = 3.0
t = np.linspace(0, duration, int(sr * duration))

# Simulate speech: vowel-like formants with variation
f0 = 150 + 20 * np.sin(2 * np.pi * 3 * t)  # Varying pitch
signal = np.zeros_like(t)
for harmonic in range(1, 8):
    signal += (0.7 ** harmonic) * np.sin(2 * np.pi * f0 * harmonic * t)
signal += 0.05 * np.random.randn(len(t))  # Add noise
signal *= np.clip(np.sin(2 * np.pi * 2 * t) * 2, 0, 1)  # Amplitude envelope

# Simple spectrogram using numpy FFT
n_fft = 512
hop = 160  # 10ms at 16kHz
n_frames = (len(signal) - n_fft) // hop
n_bins = n_fft // 2

spec = np.zeros((n_bins, n_frames))
window = np.hanning(n_fft)
for i in range(n_frames):
    frame = signal[i * hop : i * hop + n_fft] * window
    spec[:, i] = np.abs(np.fft.rfft(frame))[:-1]

# Convert to log scale (simplified mel)
spec_db = 20 * np.log10(spec + 1e-10)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(t[:4800], signal[:4800], linewidth=0.5, color="#2196F3")
axes[0].set_title("Waveform (first 300ms)")
axes[0].set_xlabel("Time (s)")

im = axes[1].imshow(spec_db, aspect="auto", origin="lower", cmap="magma",
                     extent=[0, duration, 0, sr/2])
axes[1].set_title(f"Spectrogram ({n_bins} freq bins x {n_frames} frames)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Frequency (Hz)")
plt.colorbar(im, ax=axes[1], label="dB")
plt.tight_layout()
plt.show()

print(f"Raw samples: {len(signal):,} ({duration}s at {sr}Hz)")
print(f"Spectrogram: {n_bins} x {n_frames} = {n_bins * n_frames:,} values")
print(f"Whisper frame rate: ~{n_frames / duration:.0f} frames/sec")

## 4 — Video Frame Sampling

Video understanding requires choosing **which frames to process**. Common strategies:

| Strategy | How it works | Pros | Cons |
|----------|-------------|------|------|
| **Uniform** | Every Nth frame | Simple, predictable cost | May miss key moments |
| **Keyframe** | Scene-change detection | Captures transitions | Variable cost |
| **FPS-based** | Fixed frames per second | Consistent temporal resolution | May be wasteful |

The token budget for video: `N_frames × patches_per_frame`

In [ ]:
# Video frame sampling simulation
video_duration = 60  # seconds
original_fps = 30
total_frames = video_duration * original_fps
patches_per_frame = 256  # 224x224 image, 14x14 patches

strategies = {
    "1 fps (uniform)": video_duration * 1,
    "0.5 fps (uniform)": int(video_duration * 0.5),
    "2 fps (uniform)": video_duration * 2,
    "8 keyframes": 8,
    "16 keyframes": 16,
}

print(f"Video: {video_duration}s at {original_fps}fps = {total_frames:,} total frames")
print(f"Patches per frame: {patches_per_frame}")
print()

rows = []
for name, n_frames in strategies.items():
    tokens = n_frames * patches_per_frame
    cost_ratio = tokens / 1000  # relative to 1K text tokens
    rows.append({"strategy": name, "frames": n_frames, "tokens": f"{tokens:,}",
                 "vs_1K_text": f"{cost_ratio:.1f}x"})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
names = list(strategies.keys())
tokens = [n * patches_per_frame for n in strategies.values()]
colors = ["#4CAF50" if t < 10000 else "#FF9800" if t < 20000 else "#F44336" for t in tokens]
ax.barh(names, tokens, color=colors)
ax.set_xlabel("Total visual tokens")
ax.set_title("Video Token Budget by Sampling Strategy")
for i, v in enumerate(tokens):
    ax.text(v + 200, i, f"{v:,}", va="center")
plt.tight_layout()
plt.show()

## 5 — The Unified Token Budget Calculator

Bringing it all together: a single calculator for any multimodal workload.

In [ ]:
# Comprehensive token budget calculator
def token_budget(text_tokens=0, images=None, audio_seconds=0, video_config=None):
    result = {"text": text_tokens, "image": 0, "audio": 0, "video": 0}

    if images:
        for img_res, patch_sz in images:
            result["image"] += (img_res // patch_sz) ** 2

    result["audio"] = int(audio_seconds * 50)  # ~50 frames/sec for Whisper

    if video_config:
        n_frames, frame_res, patch_sz = video_config
        result["video"] = n_frames * (frame_res // patch_sz) ** 2

    result["total"] = sum(result.values())
    return result

# Real-world workload examples
workloads = {
    "Simple image QA": token_budget(text_tokens=100, images=[(224, 14)]),
    "Document extraction (3 pages)": token_budget(text_tokens=200, images=[(896, 14)] * 3),
    "Meeting notes (5 min)": token_budget(text_tokens=150, audio_seconds=300),
    "Video summary (2 min, 1fps)": token_budget(text_tokens=100, video_config=(120, 224, 14)),
    "Full multimodal": token_budget(text_tokens=200, images=[(448, 14)], audio_seconds=60, video_config=(30, 224, 14)),
}

for name, budget in workloads.items():
    print(f"{name}:")
    for k, v in budget.items():
        if v > 0:
            print(f"  {k}: {v:,}")
    print()

# Cost comparison chart
fig, ax = plt.subplots(figsize=(10, 5))
names = list(workloads.keys())
components = ["text", "image", "audio", "video"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]
bottom = np.zeros(len(names))
for comp, color in zip(components, colors):
    vals = [workloads[n][comp] for n in names]
    ax.barh(range(len(names)), vals, left=bottom, label=comp.title(), color=color)
    bottom += np.array(vals)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel("Token Budget")
ax.set_title("Multimodal Token Budgets")
ax.legend()
plt.tight_layout()
plt.show()

## 6 — Cost Implications

Token budgets translate directly to **cost and latency**:

| Tokens | Approximate latency (T4) | Approximate cost |
|--------|-------------------------|-----------------|
| 1,000 | ~1-2s | $0.001 |
| 10,000 | ~5-10s | $0.01 |
| 50,000 | ~30-60s | $0.05 |
| 100,000+ | Minutes | $0.10+ |

**Rule of thumb**: Budget BEFORE model selection. If your workload generates 50K tokens per request, you need to either:
1. Reduce resolution/sampling rate
2. Use a smaller, faster model
3. Process in chunks with caching

## Exercises

1. **Resolution experiment**: Patch a real image at 224, 448, and 896 resolution. Plot token count vs image detail preserved (qualitative assessment).

2. **Spectrogram analysis**: Generate spectrograms for 3 different audio types (speech, music, noise). What visual patterns distinguish each?

3. **Video budget optimization**: For a 5-minute tutorial video, find the minimum sampling rate that captures all key transitions. Plot accuracy vs token cost.

In [ ]:
# Exercise starter code

# Exercise 1: Resolution experiment
for res in [224, 448, 896]:
    n_patches = (res // 14) ** 2
    print(f"Resolution {res}x{res}: {n_patches:,} patches ({n_patches / 256:.1f}x baseline)")

# Exercise 2: different "audio types" (simulated)
for audio_type, freq in [("speech", 150), ("music", 440), ("noise", None)]:
    if freq:
        sig = np.sin(2 * np.pi * freq * np.linspace(0, 1, 16000))
    else:
        sig = np.random.randn(16000)
    print(f"{audio_type}: mean amplitude = {np.abs(sig).mean():.3f}")

## Key Takeaways

| # | Insight |
|---|---------|
| 1 | Images become tokens via **patching**: (H/p × W/p) patches per image |
| 2 | Token count scales **quadratically** with resolution |
| 3 | Audio becomes tokens via **mel spectrograms** at ~50 frames/second |
| 4 | Video cost = **N_frames × patches_per_frame** — sample strategically |
| 5 | **Budget before model selection** — tokens determine cost and latency |

## What’s Next

→ [04_building_a_multimodal_benchmark_harness.ipynb](04_building_a_multimodal_benchmark_harness.ipynb) — Measuring multimodal system quality

## References

- Dosovitskiy et al., “An Image is Worth 16x16 Words” (ViT, 2020)
- Wang et al., “Qwen2-VL: Enhancing Vision-Language Model’s Perception” (2024)
- Radford et al., “Robust Speech Recognition via Large-Scale Weak Supervision” (Whisper, 2022)
- Stevens et al., “A Scale for the Measurement of the Psychological Magnitude Pitch” (Mel scale, 1937)

---

## 🧭 Navigation

⬅️ **Previous:** [02 Model Families And Modality Interfaces](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/02_model_families_and_modality_interfaces.ipynb) | ➡️ **Next:** [04 Building A Multimodal Benchmark Harness](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/04_building_a_multimodal_benchmark_harness.ipynb)